### Imports

In [ ]:
import os
import sys
import json
import time
import argparse
import warnings
import xml.etree.ElementTree as ET
import multiprocessing as mp
from pathlib import Path
from typing import Optional, Dict, List, Tuple

import numpy as np
from PIL import Image, ImageDraw

import daisy
import dask
from dask.array import coarsen, mean
from dask.diagnostics import ProgressBar
import zarr
from funlib.persistence import Array, prepare_ds, open_ds
from funlib.geometry import Roi, Coordinate
import tifffile
from tqdm import tqdm

from rtree import index
from shapely.geometry import Polygon, box

# for xml > zarr
from shapely.geometry import Polygon, box
from shapely.strtree import STRtree
from skimage.draw import polygon as draw_polygon

In [ ]:
cwd = os.getcwd()
parent_dir = os.path.abspath(os.path.join(cwd, os.pardir))

# Import OpenSlide
OPENSLIDE_PATH = os.path.join(parent_dir, 'openslide-bin-4.0.0.6-windows-x64\\bin')
print(OPENSLIDE_PATH)
if hasattr(os, 'add_dll_directory'):
    # Windows
    with os.add_dll_directory(OPENSLIDE_PATH):
        import openslide
else:
    import openslide

### FUNC: Extract Annotations

In [ ]:
def opensvs(svs_path, pyramid_level): # from rachel
    """open .svs as a dask array and retreive metadata (dimensions + resolution)"""
    slide = openslide.OpenSlide(svs_path)
    # grab resolution at micrometeres / px and convert to nm / px
    x_res = float(slide.properties["openslide.mpp-x"]) * 1000
    y_res = float(slide.properties["openslide.mpp-y"]) * 1000
    units = ("nm", "nm")
    store = tifffile.imread(svs_path, aszarr=True)
    dask_array = dask.array.from_zarr(store, pyramid_level)
    store.close()
    return dask_array, x_res, y_res, units

def svs_to_zarr(svs_path, zarr_path, offset, axis_names): # from rachel
    """convert H&E from .svs to .zarr file"""
    # open highest pyramid level
    dask_array0, x_res, y_res, units = opensvs(svs_path, 0)
    s0_shape = dask_array0.shape
    # units are natively ("nm", "nm") so no need to convert to get voxel size
    
    # convert to integer and calculate for each pyramid level
    voxel_size0 = Coordinate(int(x_res), int(y_res))
    
    # format data as funlib dataset
    raw = prepare_ds(
        zarr_path / "raw" / "s0",
        dask_array0.shape,
        offset,
        voxel_size0,
        axis_names,
        units,
        mode="w",
        dtype=np.uint8,
    )
    
    # storage info
    store_rgb = zarr.open(zarr_path / "raw" / "s0") # s0 = full resolution image at 40x magnification
    dask_array = dask_array0.rechunk(raw.data.chunksize)

    with ProgressBar():
        dask.array.store(dask_array, store_rgb)

    for i in range(1, 4): # iterate over each pyramid level: s1 (20x), s2 (10x), s3 (5x)
        # open the image file with openslide for info and tifffile as zarr
        try:
            dask_array, x_res, y_res, _ = opensvs(svs_path, i)
            # units are natively ("nm", "nm") so no need to convert to get voxel size
            
            # convert to integer and calculate for each pyramid level
            voxel_size0 = Coordinate(int(x_res), int(y_res))
            expected_shape = tuple((s0_shape[0] // 2**i, s0_shape[1] // 2**i, 3)) # downsampled shape
            print(f"expected shape: {expected_shape}")
            print(f"actual shape: {dask_array.shape}")

            # check shape is expected shape
            if dask_array.shape == expected_shape:
                print("correct shape")
                
                # format data as funlib dataset
                raw = prepare_ds(
                    zarr_path / "raw" / f"s{i}",
                    dask_array.shape,
                    offset,
                    voxel_size,
                    axis_names,
                    units,
                    mode="w",
                    dtype=np.uint8,
                )
                
                # storage info
                store_rgb = zarr.open(zarr_path / "raw" / f"s{i}")
                dask_array = dask_array.rechunk(raw.data.chunksize)

                with ProgressBar():
                    dask.array.store(dask_array, store_rgb)

            else:
                voxel_size = tuple((voxel_size0[0] * 2**i, voxel_size0[0] * 2**i))
                # format data as funlib dataset
                raw = prepare_ds(
                    zarr_path / "raw" / f"s{i}",
                    expected_shape,
                    offset,
                    voxel_size,
                    axis_names,
                    units,
                    mode="w",
                    dtype=np.uint8,
                )
                # storage info
                store_rgb = zarr.open(zarr_path / "raw" / f"s{i}")
                prev_layer = open_ds(zarr_path / "raw" / f"s{i-1}")
                print(f"chunk shape: {prev_layer.chunk_shape}")

                # mean downsampling
                factors = {0: 2, 1: 2}
                try:
                    dask_array = coarsen(mean, prev_layer.data, factors)
                except ValueError as e:
                    new_shape = tuple(
                        (
                            (prev_layer.data.shape[i] // factors[i]) * factors[i]
                            if i in factors
                            else prev_layer.data.shape[i]
                        )
                        for i in range(prev_layer.data.ndim)
                    )
                    dask_array_cropped = prev_layer.data[:new_shape[0], :new_shape[1], :new_shape[2]]
                    dask_array = coarsen(mean, dask_array_cropped, factors)
                # save to zarr
                with ProgressBar():
                    dask.array.store(dask_array, store_rgb)
        
        except TypeError as e: # if it finds an empty pyramid level, it fills it in
            print(f"for layer {i}: {e}")
            print("Generating layer")
            prev_layer = open_ds(zarr_path / "raw" / f"s{i-1}")
            voxel_size = tuple((voxel_size0[0] * 2**i, voxel_size0[0] * 2**i))
            # format data as funlib dataset
            raw = prepare_ds(
                zarr_path / "raw" / f"s{i}",
                expected_shape,
                offset,
                voxel_size,
                axis_names,
                units,
                mode="w",
                dtype=np.uint8,
            )
            # storage info
            store_rgb = zarr.open(zarr_path / "raw" / f"s{i}")
            prev_layer = open_ds(zarr_path / "raw" / f"s{i-1}")
            print(f"chunk shape: {prev_layer.chunk_shape}")
            # mean downsampling
            factors = {0: 2, 1: 2}
            try:
                dask_array = coarsen(mean, prev_layer.data, factors)
            except ValueError as e:
                new_shape = tuple(
                    (
                        (prev_layer.data.shape[i] // factors[i]) * factors[i]
                        if i in factors
                        else prev_layer.data.shape[i]
                    )
                    for i in range(prev_layer.data.ndim)
                )
                dask_array_cropped = prev_layer.data[:new_shape[0], :new_shape[1], :new_shape[2]]
                dask_array = coarsen(mean, dask_array_cropped, factors)
            # save to zarr
            with ProgressBar():
                dask.array.store(dask_array, store_rgb)
    return print("svs conversion complete")

# def (downsamples the annotation mask) --> will need to convert .xml to a .zarr

# identify blocks with valid annotations

In [ ]:
# convert .xml (Aperio Imagescope) to .zarr file
def xml_to_semantic_zarr(
    xml_path,
    svs_path,
    zarr_path,
    offset,
    axis_names=("y", "x"),
    num_pyramid_levels=4,
    chunk_size=4096,
):
    """
    Convert Aperio XML annotations to semantic segmentation Zarr
    using the same spatial metadata as svs_to_zarr().
    """

    # read slide + metadata for s0
    dask_array0, x_res, y_res, units = opensvs(svs_path, 0)
    height, width = dask_array0.shape[:2]

    voxel_size0 = Coordinate(int(x_res), int(y_res))

    # -create s0 (40x magnification)
    raw = prepare_ds(
        zarr_path / "labels" / "s0",
        (height, width),
        offset,
        voxel_size0,
        axis_names,
        units,
        mode="w",
        dtype=np.uint8,
    )

    label_store = zarr.open(zarr_path / "labels" / "s0")

    # parse .xml file
    tree = ET.parse(xml_path)
    root = tree.getroot()

    polygons = []
    class_ids = [] # corresponds to the layers in Aperio Imagescope

    for annotation in root.iter("Annotation"):
        class_id = int(annotation.attrib["Id"])

        for region in annotation.iter("Region"):
            vertices = [
                (float(v.attrib["X"]), float(v.attrib["Y"]))
                for v in region.iter("Vertex")
            ]

            if len(vertices) < 3:
                continue

            poly = Polygon(vertices)
            if not poly.is_valid:
                poly = poly.buffer(0)

            polygons.append(poly)
            class_ids.append(class_id)

    # Spatial index of annotations
    tree_index = STRtree(polygons)

    # chunked rasterization for drawing annotations
    for y0 in range(0, height, chunk_size):
        for x0 in range(0, width, chunk_size):

            y1 = min(y0 + chunk_size, height)
            x1 = min(x0 + chunk_size, width)

            tile_mask = np.zeros((y1 - y0, x1 - x0), dtype=np.uint8)
            tile_box = box(x0, y0, x1, y1)

            candidate_indices = tree_index.query(tile_box)

            for idx in candidate_indices:
                poly = polygons[idx]

                if not poly.intersects(tile_box):
                    continue

                clipped = poly.intersection(tile_box)
                if clipped.is_empty:
                    continue

                if clipped.geom_type != "Polygon":
                    continue

                coords = np.array(clipped.exterior.coords)

                rr, cc = draw_polygon(
                    coords[:, 1] - y0,
                    coords[:, 0] - x0,
                    tile_mask.shape,
                )

                class_id = class_ids[idx]
                tile_mask[rr, cc] = class_id # fill the polygon with class id

                label_store[y0:y1, x0:x1] = tile_mask

    print("labels/s0 written")

    # pyramid generation
    prev = dask.array.from_zarr(zarr_path / "labels" / "s0")

    for level in range(1, num_pyramid_levels):

        voxel_size = Coordinate(
            int(x_res * (2**level)),
            int(y_res * (2**level)),
        )

        down = dask.array.coarsen(
            np.max,
            prev,
            {0: 2, 1: 2},
            trim_excess=True # drops last row if the pixel dimensions are odd
        )

        # create funlib dataset for this level
        raw = prepare_ds(
            zarr_path / "labels" / f"s{level}",
            down.shape,
            offset,
            voxel_size,
            axis_names,
            units,
            mode="w",
            dtype=np.uint8,
        )

        store = zarr.open(zarr_path / "labels" / f"s{level}")

        with ProgressBar():
            dask.array.store(down, store)

        prev = down

        print(f"labels/s{level} written")

    print("Semantic XML → Zarr conversion complete")


In [ ]:
def save_npz_daisy(
    img_10x: Array,
    anno_mask_10x: Array
):
    """
    Uses daisy to save 224x224 pixel blocks as a .npz from annotated H&E WSIs (.zarr)
    """
    def save_npz_block(block: daisy.Block):
        img_data = img_10x[block.read_roi] # read image (will be in a funlib array format: Array)
        # convert img_data: Array to a np.array for saving it as a .npz
        anno_mask_data = anno_mask_10x[block.read_roi]
        # convert anno_mask_data: Array to a np.array for saving it as a .npz

        # save + format both the img and anno_mask into a .npz using naming convention

    block_roi = Roi((0, 0), (224, 224)) * img_10x.voxel_size

    save_npz_task = daisy.Task(
        "blockwise_save",
        total_roi=img_10x.roi, # dictates the total size for both annotations + img
        read_roi=block_roi,
        write_roi=block_roi,
        read_write_conflict=False,
        num_workers=2,
        process_function=save_npz_block,
    )
    daisy.run_blockwise(tasks=[save_npz_task], multiprocessing=False)
    return

In [ ]:
# run daisy on valid blocks

### Execution

In [ ]:
# convert .svs to .zarr
svs_path = Path(r"\\Mittal-MRL-NAS\Research Data\DCIS_project\Lisa\testing_transunet_training_patch_extraction\2015003_H&E.svs")
zarr_path = Path(r"\\Mittal-MRL-NAS\Research Data\DCIS_project\Lisa\testing_transunet_training_patch_extraction\2015003_H&E_img.zarr")
offset = Coordinate(0, 0)
axis_names = ['y', 'x', 'c^']

svs_to_zarr(svs_path, 
            zarr_path, 
            offset, 
            axis_names
            )

# TODO: note that the .zarr is huge??

In [ ]:
xml_path = Path(r"\\Mittal-MRL-NAS\Research Data\DCIS_project\Lisa\testing_transunet_training_patch_extraction\2015003_H&E.xml")
svs_path = Path(r"\\Mittal-MRL-NAS\Research Data\DCIS_project\Lisa\testing_transunet_training_patch_extraction\2015003_H&E.svs")
zarr_path = Path(r"\\Mittal-MRL-NAS\Research Data\DCIS_project\Lisa\testing_transunet_training_patch_extraction\2015003_H&E_xml.zarr")
offset = Coordinate(0, 0)

xml_to_semantic_zarr(xml_path,
                     svs_path,
                     zarr_path,
                     offset
                    )